# Shared Initialization Experiment

In Session 3, naive FedAvg aggregation reduced test accuracy from 78.75% and 73.75% to 47.5%.

One possible explanation is that both clients started from different random initializations and converged to different regions of the quantum parameter landscape.

This experiment investigates whether starting from the same initial parameters improves aggregation performance.

## Recreating Session 1 Setup

In [1]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss

from qiskit_algorithms.optimizers import COBYLA

# --------------------------------------------------
# Dataset
# --------------------------------------------------

X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------------
# Federated Split (same as Session 1)
# --------------------------------------------------

client1_idx = np.where(y_train == 0)[0][:128]
client1_idx = np.concatenate([
    client1_idx,
    np.where(y_train == 1)[0][:32]
])

client2_idx = np.where(y_train == 0)[0][128:158]
client2_idx = np.concatenate([
    client2_idx,
    np.where(y_train == 1)[0][32:160]
])

X_client1 = X_train[client1_idx]
y_client1 = y_train[client1_idx]

X_client2 = X_train[client2_idx]
y_client2 = y_train[client2_idx]

# --------------------------------------------------
# Convert Labels to {-1, +1}
# --------------------------------------------------

y_client1_q = 2 * y_client1 - 1
y_client2_q = 2 * y_client2 - 1
y_test_q = 2 * y_test - 1

# --------------------------------------------------
# Quantum Circuit
# --------------------------------------------------

num_qubits = 4

x_params = ParameterVector("x", 2)
theta = ParameterVector("θ", 8)

qc = QuantumCircuit(num_qubits)

qc.ry(x_params[0], 0)
qc.ry(x_params[1], 1)

for i in range(num_qubits):
    qc.ry(theta[2*i], i)
    qc.rz(theta[2*i + 1], i)

qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)

observable = SparsePauliOp.from_list([
    ("ZZZZ", 1)
])

# --------------------------------------------------
# QNN
# --------------------------------------------------

qnn = EstimatorQNN(
    circuit=qc,
    input_params=x_params,
    weight_params=theta,
    observables=observable
)

print("Client 1 samples:", len(X_client1))
print("Client 2 samples:", len(X_client2))
print("Test samples:", len(X_test))

print("\nQNN ready.")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Client 1 samples: 160
Client 2 samples: 158
Test samples: 80

QNN ready.


In [2]:
shared_initial_weights = np.random.rand(8)

print("Shared Initial Weights:")
print(shared_initial_weights)

Shared Initial Weights:
[0.6665394  0.77237282 0.06018823 0.09506984 0.8922146  0.26490982
 0.99573136 0.48060534]


### Client 1 Training (Shared Initialization)

In [3]:
classifier_client1 = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=100),
    loss=CrossEntropyLoss(),
    one_hot=False,
    initial_point=shared_initial_weights
)

classifier_client1.fit(
    X_client1,
    y_client1_q
)

print("Client 1 training complete!")

Client 1 training complete!


### Client 2 Training (Shared Initialization)

In [4]:
classifier_client2 = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=100),
    loss=CrossEntropyLoss(),
    one_hot=False,
    initial_point=shared_initial_weights
)

classifier_client2.fit(
    X_client2,
    y_client2_q
)

print("Client 2 training complete!")

Client 2 training complete!


In [5]:
print("Client 1 Weights:")
print(classifier_client1.weights)

print("\nClient 2 Weights:")
print(classifier_client2.weights)

Client 1 Weights:
[ 2.40943594 -0.05749264  1.62613288  0.65715057  1.89151744  0.31999586
 -0.07415539  0.53583397]

Client 2 Weights:
[0.66283091 0.52603229 0.957417   0.09506537 0.89228536 0.26489247
 0.9272971  0.48101243]


In [6]:
shared_global_weights = (
    classifier_client1.weights +
    classifier_client2.weights
) / 2

print("FedAvg Weights:")
print(shared_global_weights)

FedAvg Weights:
[1.53613342 0.23426983 1.29177494 0.37610797 1.3919014  0.29244416
 0.42657085 0.5084232 ]


# Evaluating Shared-Initialization FedAvg

In [7]:
train_acc1 = classifier_client1.score(
    X_client1,
    y_client1_q
)

test_acc1 = classifier_client1.score(
    X_test,
    y_test_q
)

train_acc2 = classifier_client2.score(
    X_client2,
    y_client2_q
)

test_acc2 = classifier_client2.score(
    X_test,
    y_test_q
)

print("Client 1 Train Accuracy:", train_acc1)
print("Client 1 Test Accuracy :", test_acc1)

print("\nClient 2 Train Accuracy:", train_acc2)
print("Client 2 Test Accuracy :", test_acc2)

Client 1 Train Accuracy: 0.9375
Client 1 Test Accuracy : 0.725

Client 2 Train Accuracy: 0.930379746835443
Client 2 Test Accuracy : 0.775


In [8]:
predictions = []

for sample in X_test:

    output = qnn.forward(
        input_data=sample,
        weights=shared_global_weights
    )

    pred = 1 if output[0][0] >= 0 else -1

    predictions.append(pred)

predictions = np.array(predictions)

shared_fedavg_accuracy = np.mean(
    predictions == y_test_q
)

print("Shared-Init FedAvg Accuracy:")
print(shared_fedavg_accuracy)

Shared-Init FedAvg Accuracy:
0.7625
